# 02 · Estudio de eventos: ¿cuánta lluvia hubo cuando bajaron los huaicos?

Pregunta: con los huaicos con día exacto de Chosica y Chaclacayo desde 1983, ¿qué tan extrema fue la lluvia esos días respecto de toda la temporada de lluvias (diciembre a abril)? ¿Y a cuántas alarmas falsas habría llevado usarla como señal?

Eventos: `catalogo/` (cronología del IGP, que cita El Comercio y La República, más el catálogo previo). Lluvia: Open-Meteo (ERA5), en Chosica y en la cuenca alta. `uv run descargar-datos --solo open-meteo-horario`.

**Límites.** El catálogo solo tiene eventos reportados: los días extremos sin evento pueden ser huaicos no registrados, así que las alarmas por año son una cota superior de falsas alarmas, no una tasa medida. Es lluvia de reanálisis, que aplana los extremos. No es un sistema de alerta.

In [ ]:
from pathlib import Path

import pandas as pd

from illapa_research.catalogo import cargar_todos
from illapa_research.estudio_eventos import (
    cargar_horario,
    diario,
    percentil_evento,
    puntaje,
    tasa_aciertos,
    temporada,
)

horario = cargar_horario(Path("../datos/open_meteo_horario_estudio.json"))
D = diario(horario["chosica"], "ch").join(diario(horario["cuenca-alta"], "al"))
temp = temporada(D, hasta=2025)
# One row per date: the same day can appear in both catalogs.
por_fecha = {}
for e in cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True):
    if e.fecha.year >= 1983:
        por_fecha.setdefault(e.fecha, []).extend(e.quebradas)
eventos = [(f, sorted(set(q))) for f, q in sorted(por_fecha.items())]
print(f"{len(eventos)} fechas con huaico y día exacto; {len(temp)} días de temporada ({temp.index.year.nunique()} años)")

## Percentil de la lluvia en cada evento

100 = el día más lluvioso de toda la serie. `p1d` es la lluvia del día, `p3d` la de 3 días, `p7d` la de los 7 días anteriores; `ch` es Chosica y `al` la cuenca alta.

In [ ]:
cols = ["ch_p1d", "ch_p3d", "ch_p7d", "al_p1d", "al_p3d", "al_p7d"]
filas = []
for fecha, quebradas in eventos:
    f = {"fecha": fecha, "quebradas": "; ".join(quebradas)[:30]}
    f |= {c: round(percentil_evento(D, temp, fecha, c)) for c in cols}
    filas.append(f)
tabla = pd.DataFrame(filas)
display(tabla.style.background_gradient(subset=cols, cmap="YlOrRd", vmin=50, vmax=100))
print("eventos con p1d >= p95 en Chosica o en la cuenca alta:",
      int(((tabla.ch_p1d >= 95) | (tabla.al_p1d >= 95)).sum()), "de", len(tabla))

## ¿Sirve como señal? Aciertos contra alarmas por año

Puntaje diario = el percentil más alto entre la lluvia de 1 y 3 días en ambos puntos. Con cada umbral: cuántos eventos se capturan (el día o el día anterior) y cuántos días de alarma por año habría.

In [ ]:
score = puntaje(temp, ["ch_p1d", "al_p1d", "ch_p3d", "al_p3d"])
fechas = [f for f, _ in eventos]
res = []
for u in (90, 95, 97, 99):
    a, por_anio = tasa_aciertos(score, fechas, u)
    res.append({"umbral (percentil)": u, "eventos capturados": f"{a}/{len(fechas)}", "días de alarma por año": round(por_anio, 1)})
pd.DataFrame(res)

## Días extremos sin evento: pistas para la hemeroteca

Son los días más lluviosos que no están en el catálogo. Cada uno es una pregunta: ¿hubo huaico y no lo registramos, o la lluvia del reanálisis no se tradujo en uno? Se confirman o descartan contra prensa e informes (INDECI, COEN, CENEPRED).

In [ ]:
extremos = score[score >= 99.5]
sin_evento = [d for d in extremos.index if not any(abs((d.date() - f).days) <= 2 for f in fechas)]
print(f"{len(extremos)} días >= p99.5; {len(sin_evento)} sin evento catalogado (±2 días)")
pd.Series([d.date() for d in sin_evento]).groupby(pd.Series([d.year for d in sin_evento])).apply(list).head(40)